# 14.03 · Voice agents

**Chapter:** Multimodal AI · **Milestone:** M7

## Goals
- [ ] Compare the two architectures: **cascaded** (STT → LLM → TTS) vs **speech-to-speech** realtime models
- [ ] Build a cascaded voice turn from scratch over the RAG index, and measure every stage of the **latency budget**
- [ ] Cut time-to-first-audio with streaming: sentence-level TTS while the LLM is still writing
- [ ] Endpointing (when did the user stop talking?), barge-in, and voice-specific prompting
- [ ] Know the frameworks: Pipecat, LiveKit Agents, and realtime APIs (OpenAI Realtime, Gemini Live)

## Resources
- [Pipecat docs](https://docs.pipecat.ai/) · [LiveKit Agents](https://docs.livekit.io/agents/) — free/OSS
- [Voice AI & Voice Agents – an illustrated primer](https://voiceaiandvoiceagents.com/) (Daily/Pipecat) — free
- [OpenAI Realtime API](https://platform.openai.com/docs/guides/realtime) · [Gemini Live API](https://ai.google.dev/gemini-api/docs/live) — paid / free tier
- [Silero VAD](https://github.com/snakers4/silero-vad) · [Pipecat smart-turn](https://github.com/pipecat-ai/smart-turn) — free

People expect a reply ~0.5–1 s after they stop talking (human turn gaps are ~200 ms). Every stage adds latency:

| Stage | Typical | Notes |
|---|---|---|
| Endpointing (detect end of turn) | 200–800 ms | silence timeout or a turn-detection model |
| STT final transcript | 100–500 ms | streaming STT hides most of it |
| LLM time to first token | 200–800 ms | small/fast models, short prompts, prompt caching |
| TTS time to first audio | 100–300 ms | streaming TTS |
| Network + audio buffers | 50–200 ms | WebRTC, regional servers |

**Speech-to-speech** models (OpenAI Realtime, Gemini Live, open: Moshi, Sesame CSM, Qwen-Omni) take audio in and produce audio out in one
model: lowest latency, keeps tone/emotion, handles interruptions natively — but harder to control, to evaluate, and to plug into your own RAG/tools.
**Cascaded** pipelines are modular (swap any part), debuggable (you have the text), and work with any LLM. Most production agents are cascaded.

In [ ]:
import re
import subprocess
import time

import mlx_whisper
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from kokoro import KPipeline

from aieng import DATA, rag
from aieng.llm import available, client

AUDIO = DATA / "audio"
AUDIO.mkdir(exist_ok=True)
SR = 16_000
STT_MODEL = "mlx-community/whisper-large-v3-turbo"

if "groq" in available():  # Groq: very low TTFT. Fallback: local Ollama.
    llm, _ = client("groq"); LLM_MODEL, EXTRA = "openai/gpt-oss-20b", {"reasoning_effort": "low"}  # model ids change: check console.groq.com
else:
    llm, LLM_MODEL = client("ollama"); EXTRA = {"reasoning_effort": "none"}
tts = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")
print("LLM:", LLM_MODEL)

## 1. The user's turn (simulated)
We synthesise the user's spoken question with `say`, plus 1.5 s of trailing silence, the way a microphone stream would look.

In [ ]:
def user_says(text, voice="Daniel"):
    path = AUDIO / "user.wav"
    subprocess.run(["say", "-v", voice, "-o", str(path), f"--data-format=LEI16@{SR}", text], check=True)
    audio, _ = sf.read(path, dtype="float32")
    return np.concatenate([np.zeros(SR // 2, np.float32), audio, np.zeros(int(1.5 * SR), np.float32)])


mic = user_says("When was Abraham Lincoln born, and where?")
display(Audio(mic, rate=SR))

## 2. Endpointing: when did the user stop talking?
The simplest endpointer: frame energy above a threshold = speech; *N* ms of silence after speech = end of turn. Too short a timeout and you
interrupt people mid-thought; too long and the agent feels slow. Production systems use a VAD model (Silero) plus a **semantic turn detector**
that looks at the words ("and where…" → not finished).

In [ ]:
def endpoint(audio, frame_ms=20, silence_ms=500, threshold=0.01):
    """Return (speech_end_s, endpoint_s): when speech stopped, and when we are confident the turn is over."""
    n = SR * frame_ms // 1000
    rms = np.sqrt(np.mean(audio[: len(audio) // n * n].reshape(-1, n) ** 2, axis=1))
    speech = rms > threshold
    last, silent = None, 0
    for i, s in enumerate(speech):
        if s:
            last, silent = i, 0
        elif last is not None:
            silent += 1
            if silent * frame_ms >= silence_ms:
                return (last + 1) * frame_ms / 1000, (i + 1) * frame_ms / 1000
    return len(audio) / SR, len(audio) / SR


speech_end, turn_end = endpoint(mic)
print(f"speech ends at {speech_end:.2f}s, turn detected at {turn_end:.2f}s -> endpointing cost {(turn_end - speech_end) * 1000:.0f} ms")

## 3. A cascaded turn, timed stage by stage
Voice prompts differ from chat prompts: short, no markdown or lists (they'd be read aloud), numbers and abbreviations written to be spoken.

In [ ]:
VOICE_PROMPT = """You are a voice assistant. Answer in at most two short spoken sentences, using only the sources.
No markdown, no lists, no citations, no brackets. If the sources don't contain the answer, say you don't know.

Sources:
{sources}

Question: {question}"""


def speak(text, voice="af_heart"):
    return np.concatenate([a.numpy() if hasattr(a, "numpy") else a for _, _, a in tts(text, voice=voice)])


speak("warm up"); mlx_whisper.transcribe(mic[:SR], path_or_hf_repo=STT_MODEL); rag.search("warm up")  # load everything once, like a server would


def turn_blocking(mic):
    t = {}
    start = time.perf_counter()
    _, turn_end = endpoint(mic)
    question = mlx_whisper.transcribe(mic[: int(turn_end * SR)], path_or_hf_repo=STT_MODEL, language="en")["text"].strip()
    t["stt"] = time.perf_counter() - start
    hits = rag.search(question, k=3)
    sources = "\n\n".join(text for _, _, text in hits)
    t["retrieval"] = time.perf_counter() - start - t["stt"]
    r = llm.chat.completions.create(model=LLM_MODEL, messages=[{"role": "user", "content": VOICE_PROMPT.format(sources=sources, question=question)}],
                                    temperature=0, max_tokens=400, **EXTRA)  # headroom: reasoning tokens count too
    answer = r.choices[0].message.content.strip()
    t["llm"] = time.perf_counter() - start - t["stt"] - t["retrieval"]
    audio = speak(answer)
    t["tts"] = time.perf_counter() - start - sum(t.values())
    return question, answer, audio, t


question, answer, reply, t = turn_blocking(mic)
print(f"heard: {question!r}\nsaid : {answer!r}")
print(" | ".join(f"{k} {v * 1000:.0f} ms" for k, v in t.items()), f"| first audio after {sum(t.values()) * 1000:.0f} ms (+ endpointing)")
display(Audio(reply, rate=24_000))

## 4. Streaming: speak the first sentence while the LLM writes the rest
The user only needs the *first* audio chunk to start listening. Stream LLM tokens, cut at sentence boundaries, and send each sentence to TTS
immediately (real systems also stream STT and TTS audio frames). Measure **time to first audio**, the number users feel.

In [ ]:
def turn_streaming(mic):
    start = time.perf_counter()
    _, turn_end = endpoint(mic)
    question = mlx_whisper.transcribe(mic[: int(turn_end * SR)], path_or_hf_repo=STT_MODEL, language="en")["text"].strip()
    sources = "\n\n".join(text for _, _, text in rag.search(question, k=3))
    stream = llm.chat.completions.create(model=LLM_MODEL, messages=[{"role": "user", "content": VOICE_PROMPT.format(sources=sources, question=question)}],
                                         temperature=0, max_tokens=400, stream=True, **EXTRA)
    buffer, chunks, first_audio = "", [], None
    for ch in stream:
        if ch.choices and ch.choices[0].delta.content:
            buffer += ch.choices[0].delta.content
            *done, buffer = re.split(r"(?<=[.!?])\s+", buffer)  # complete sentences out, remainder stays in the buffer
            for sentence in done:
                chunks.append(speak(sentence))  # in a real app: push to the audio output queue right away
                first_audio = first_audio or time.perf_counter() - start
    if buffer.strip():
        chunks.append(speak(buffer))
        first_audio = first_audio or time.perf_counter() - start
    return np.concatenate(chunks), first_audio, time.perf_counter() - start


for q in ["When was Abraham Lincoln born, and where?", "What is the capital of Uruguay and what is it known for?"]:
    mic = user_says(q)
    *_, t = turn_blocking(mic)
    audio, first, total = turn_streaming(mic)
    print(f"{q[:45]:45} blocking first audio {sum(t.values()) * 1000:5.0f} ms | streaming first audio {first * 1000:5.0f} ms (done {total * 1000:.0f} ms)")
display(Audio(audio, rate=24_000))

Streaming only helps when there is something to overlap: a one-sentence answer is spoken only after the whole sentence exists, so it can even
be a bit slower (more requests, smaller TTS batches). Answers of two or more sentences start talking much earlier. Some systems cut the first
chunk at a comma, or have the LLM start with a short sentence on purpose.

Where does the rest of the time go? Look at the stage table from §3: on a laptop, STT on the full utterance is often the biggest stage.
Streaming STT (transcribing while the user speaks) removes most of it; a smaller/faster LLM and a hosted TTS remove more.
Budget on paper first, then measure — exactly like 13.06.

## 5. What makes voice agents hard
- **Barge-in**: the user interrupts → stop TTS playback immediately, cancel the LLM stream, and keep only what was actually spoken in history.
- **Backchannels** ("mm-hmm", "right") shouldn't trigger a new turn; filler ("um… let me think") shouldn't end one.
- **Errors you can't see**: a wrong transcript silently becomes a wrong answer. Log transcripts and audio (with consent), eval on real calls.
- **Tool calls** take time: say something ("Let me check that…") while the tool runs.
- **Telephony**: 8 kHz audio, codecs, DTMF; Twilio/Telnyx/Daily/LiveKit SIP bridge phone calls to your pipeline.

## 6. Frameworks
**Pipecat** composes the pipeline from processors that stream frames (audio, text, control) and handles VAD, interruptions and transports
(WebRTC via Daily, WebSockets, telephony):
```python
pipeline = Pipeline([
    transport.input(),                       # mic audio frames (WebRTC)
    stt,                                     # e.g. DeepgramSTTService / WhisperSTTService (local)
    context_aggregator.user(),
    llm,                                     # OpenAILLMService(base_url=...) — any OpenAI-compatible model
    tts,                                     # CartesiaTTSService / KokoroTTSService / ...
    transport.output(),                      # speaker audio frames
    context_aggregator.assistant(),
])
```
**LiveKit Agents** does the same around LiveKit's WebRTC rooms (`AgentSession(stt=..., llm=..., tts=..., vad=silero.VAD.load(), turn_detection=...)`),
with a strong turn-detector model and telephony. **Realtime APIs**: one WebSocket/WebRTC session, audio in → audio out, tools supported;
good for the most natural conversation if you accept a single provider.

## 7. (Optional) Talk to it
Records 5 s from your microphone, runs one streaming turn, plays the answer. Needs mic permission for your terminal/IDE.

In [ ]:
LIVE = False
if LIVE:
    import sounddevice as sd
    print("Speak now (5 s)...")
    rec = sd.rec(5 * SR, samplerate=SR, channels=1, dtype="float32"); sd.wait()
    audio, first, total = turn_streaming(np.concatenate([rec[:, 0], np.zeros(SR, np.float32)]))
    print(f"first audio {first * 1000:.0f} ms"); sd.play(audio, 24_000); sd.wait()

## Exercises
1. Build a voice assistant that answers questions from your **own** RAG index (M3): swap `rag.search` for your retriever, then run it live (§7).
2. Implement barge-in in `turn_streaming`: play sentences with `sounddevice` in a thread, keep recording, and stop playback when the mic energy
   rises above the threshold. Record what was actually spoken into the chat history.
3. Replace the energy endpointer with Silero VAD (`pip install silero-vad`) and compare false endpoints on speech with pauses
   ("I'd like to book… um… a table for two").
4. Run a Pipecat example (`pipecat-ai` with Groq STT + Groq LLM + Kokoro/Cartesia TTS over a local WebRTC transport) and compare its latency with yours.

In [ ]:
# your code here